<a href="https://colab.research.google.com/github/mdzikrim/MachineLearningClass/blob/main/Chapter_19_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import time

print("TensorFlow version:", tf.__version__)

# =============================================================================
# 1. PERSIAPAN DATA
# =============================================================================

def load_and_prepare_data():
    """Load dan persiapkan data California Housing"""
    print("Loading California Housing dataset...")

    # Load dataset
    housing = fetch_california_housing()
    X, y = housing.data, housing.target

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Training set shape: {X_train_scaled.shape}")
    print(f"Test set shape: {X_test_scaled.shape}")

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# =============================================================================
# 2. MEMBUAT MODEL DENGAN KERAS FUNCTIONAL API
# =============================================================================

def create_model(input_shape):
    """Membuat model neural network dengan Functional API"""

    # Input layer
    inputs = tf.keras.layers.Input(shape=input_shape, name="features")

    # Hidden layers
    hidden1 = tf.keras.layers.Dense(30, activation="relu", name="hidden1")(inputs)
    hidden2 = tf.keras.layers.Dense(30, activation="relu", name="hidden2")(hidden1)

    # Output layer
    outputs = tf.keras.layers.Dense(1, name="output")(hidden2)

    # Create model
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="housing_model")

    return model

# =============================================================================
# 3. CUSTOM TRAINING LOOP (Advanced Training)
# =============================================================================

def custom_training_step(model, optimizer, loss_fn, X_batch, y_batch):
    """Custom training step menggunakan GradientTape"""

    with tf.GradientTape() as tape:
        # Forward pass
        predictions = model(X_batch, training=True)
        # Compute loss
        loss = loss_fn(y_batch, predictions)

    # Compute gradients
    gradients = tape.gradient(loss, model.trainable_variables)

    # Apply gradients
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

def train_with_custom_loop(model, X_train, y_train, X_val, y_val, epochs=10, batch_size=32):
    """Training dengan custom loop untuk kontrol yang lebih baik"""

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    loss_fn = tf.keras.losses.MeanSquaredError()

    # Convert to TensorFlow datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.shuffle(1000).batch(batch_size)

    history = {'loss': [], 'val_loss': []}

    print("Starting custom training loop...")

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        # Training
        epoch_loss = 0
        num_batches = 0

        for X_batch, y_batch in train_dataset:
            loss = custom_training_step(model, optimizer, loss_fn, X_batch, y_batch)
            epoch_loss += loss
            num_batches += 1

        avg_loss = epoch_loss / num_batches

        # Validation
        val_predictions = model(X_val, training=False)
        val_loss = loss_fn(y_val, val_predictions)

        history['loss'].append(float(avg_loss))
        history['val_loss'].append(float(val_loss))

        print(f"Loss: {avg_loss:.4f} - Val Loss: {val_loss:.4f}")

    return history

# =============================================================================
# 4. MODEL CHECKPOINTING DAN CALLBACKS
# =============================================================================

def setup_callbacks():
    """Setup berbagai callbacks untuk training"""

    callbacks = [
        # Model checkpoint
        tf.keras.callbacks.ModelCheckpoint(
            filepath="best_model.h5",
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        # Early stopping
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),

        # Learning rate reduction
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),

        # TensorBoard logging
        tf.keras.callbacks.TensorBoard(
            log_dir="./logs",
            histogram_freq=1,
            write_graph=True
        )
    ]

    return callbacks

# =============================================================================
# 5. DISTRIBUTED TRAINING SIMULATION
# =============================================================================

def simulate_distributed_training():
    """Simulasi distributed training dengan MirroredStrategy"""

    print("Simulating distributed training...")

    # Create strategy
    strategy = tf.distribute.MirroredStrategy()
    print(f"Number of devices: {strategy.num_replicas_in_sync}")

    # Create model within strategy scope
    with strategy.scope():
        distributed_model = create_model((8,))
        distributed_model.compile(
            optimizer='adam',
            loss='mse',
            metrics=['mae']
        )

    print("Distributed model created successfully!")
    return distributed_model

# =============================================================================
# 6. MODEL SERVING DAN DEPLOYMENT
# =============================================================================

def save_model_for_serving(model, model_name="housing_model"):
    """Save model dalam format yang siap untuk serving"""

    # Save dalam format SavedModel (recommended untuk production)
    tf.saved_model.save(model, f"./saved_models/{model_name}")
    print(f"Model saved to ./saved_models/{model_name}")

    # Save dalam format HDF5 (untuk backup)
    model.save(f"./saved_models/{model_name}.h5")
    print(f"Model saved to ./saved_models/{model_name}.h5")

def load_and_serve_model(model_path):
    """Load model dan simulasi serving"""

    print(f"Loading model from {model_path}")
    loaded_model = tf.saved_model.load(model_path)

    # Simulasi prediction request
    sample_input = np.random.randn(1, 8).astype(np.float32)

    # Make prediction
    prediction = loaded_model(sample_input)
    print(f"Sample prediction: {prediction.numpy()[0][0]:.2f}")

    return loaded_model

# =============================================================================
# 7. PERFORMANCE MONITORING
# =============================================================================

def benchmark_model_performance(model, X_test, num_runs=100):
    """Benchmark performa model untuk monitoring"""

    print("Benchmarking model performance...")

    # Warm up
    _ = model(X_test[:10])

    # Benchmark inference time
    times = []
    for _ in range(num_runs):
        start_time = time.time()
        _ = model(X_test)
        end_time = time.time()
        times.append(end_time - start_time)

    avg_time = np.mean(times)
    throughput = len(X_test) / avg_time

    print(f"Average inference time: {avg_time*1000:.2f} ms")
    print(f"Throughput: {throughput:.0f} samples/second")

    return {'avg_time': avg_time, 'throughput': throughput}

# =============================================================================
# 8. MAIN EXECUTION
# =============================================================================

def main():
    """Fungsi utama untuk menjalankan semua demonstrasi"""

    print("=" * 60)
    print("CHAPTER 19 DEMO: TRAINING AND DEPLOYING AT SCALE")
    print("=" * 60)

    # 1. Load data
    X_train, X_test, y_train, y_test, scaler = load_and_prepare_data()

    # Split training data untuk validation
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )

    # 2. Create model
    print("\n" + "="*50)
    print("CREATING MODEL")
    print("="*50)

    model = create_model((X_train.shape[1],))
    model.summary()

    # 3. Standard training
    print("\n" + "="*50)
    print("STANDARD TRAINING")
    print("="*50)

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    callbacks = setup_callbacks()

    history_standard = model.fit(
        X_train_split, y_train_split,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

    # 4. Custom training loop
    print("\n" + "="*50)
    print("CUSTOM TRAINING LOOP")
    print("="*50)

    custom_model = create_model((X_train.shape[1],))
    history_custom = train_with_custom_loop(
        custom_model, X_train_split, y_train_split, X_val, y_val,
        epochs=10, batch_size=32
    )

    # 5. Distributed training simulation
    print("\n" + "="*50)
    print("DISTRIBUTED TRAINING SIMULATION")
    print("="*50)

    distributed_model = simulate_distributed_training()

    # 6. Model evaluation
    print("\n" + "="*50)
    print("MODEL EVALUATION")
    print("="*50)

    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test MAE: {test_mae:.4f}")

    # 7. Save model for serving
    print("\n" + "="*50)
    print("SAVING MODEL FOR SERVING")
    print("="*50)

    os.makedirs("./saved_models", exist_ok=True)
    save_model_for_serving(model, "housing_predictor")

    # 8. Load and serve model
    print("\n" + "="*50)
    print("LOADING AND SERVING MODEL")
    print("="*50)

    try:
        served_model = load_and_serve_model("./saved_models/housing_predictor")
    except Exception as e:
        print(f"Model serving simulation failed: {e}")

    # 9. Performance benchmarking
    print("\n" + "="*50)
    print("PERFORMANCE BENCHMARKING")
    print("="*50)

    performance_stats = benchmark_model_performance(model, X_test)

    # 10. Plot training history
    print("\n" + "="*50)
    print("PLOTTING TRAINING RESULTS")
    print("="*50)

    plt.figure(figsize=(12, 4))

    # Standard training history
    plt.subplot(1, 2, 1)
    plt.plot(history_standard.history['loss'], label='Training Loss')
    plt.plot(history_standard.history['val_loss'], label='Validation Loss')
    plt.title('Standard Training')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Custom training history
    plt.subplot(1, 2, 2)
    plt.plot(history_custom['loss'], label='Training Loss')
    plt.plot(history_custom['val_loss'], label='Validation Loss')
    plt.title('Custom Training Loop')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("DEMO COMPLETED SUCCESSFULLY!")
    print("="*60)

# Jalankan demonstrasi
if __name__ == "__main__":
    main()

# =============================================================================
# ADDITIONAL UTILITIES FOR PRODUCTION
# =============================================================================

def create_inference_function(model_path):
    """Membuat fungsi inference yang optimized untuk production"""

    @tf.function
    def inference_fn(x):
        """Optimized inference function"""
        return model(x, training=False)

    model = tf.keras.models.load_model(model_path)
    return inference_fn

def setup_monitoring():
    """Setup monitoring untuk production deployment"""

    print("Setting up production monitoring...")

    # Metrics yang perlu dimonitor:
    metrics = {
        'latency': 'Average response time',
        'throughput': 'Requests per second',
        'error_rate': 'Percentage of failed requests',
        'resource_usage': 'CPU and memory usage',
        'model_accuracy': 'Model performance metrics'
    }

    for metric, description in metrics.items():
        print(f"- {metric}: {description}")

    return metrics

print("\nKode siap dijalankan! Gunakan main() untuk memulai demonstrasi.")
print("Pastikan Anda memiliki TensorFlow, scikit-learn, dan matplotlib terinstall.")

TensorFlow version: 2.19.0
CHAPTER 19 DEMO: TRAINING AND DEPLOYING AT SCALE
Loading California Housing dataset...
Training set shape: (16512, 8)
Test set shape: (4128, 8)

CREATING MODEL


Model: "housing_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden1 (Dense)                 │ (None, 30)             │           270 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden2 (Dense)                 │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,231 (4.81 KB)

 Trainable params: 1,231 (4.81 KB)

...

413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3229 - mae: 0.3968 - val_loss: 0.3518 - val_mae: 0.4154 - learning_rate: 0.0010
Epoch 14/20
393/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3200 - mae: 0.3912
Epoch 14: val_loss improved from 0.35176 to 0.34790, saving model to best_model.h5


413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3197 - mae: 0.3911 - val_loss: 0.3479 - val_mae: 0.4061 - learning_rate: 0.0010
Epoch 15/20
409/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3058 - mae: 0.3835
Epoch 15: val_loss did not improve from 0.34790
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3058 - mae: 0.3836 - val_loss: 0.3610 - val_mae: 0.4050 - learning_rate: 0.0010
Epoch 16/20
403/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3091 - mae: 0.3875
Epoch 16: val_loss improved from 0.34790 to 0.33177, saving model to best_model.h5


413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3089 - mae: 0.3874 - val_loss: 0.3318 - val_mae: 0.4000 - learning_rate: 0.0010
Epoch 17/20
412/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3113 - mae: 0.3855
Epoch 17: val_loss did not improve from 0.33177
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3113 - mae: 0.3855 - val_loss: 0.3735 - val_mae: 0.3999 - learning_rate: 0.0010
Epoch 18/20
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3134 - mae: 0.3888
Epoch 18: val_loss did not improve from 0.33177
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3134 - mae: 0.3888 - val_loss: 0.3487 - val_mae: 0.4112 - learning_rate: 0.0010
Epoch 19/20
401/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2913 - mae: 0.3755
Epoch 19: val_loss did not improve from 0.33177
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2914 - mae: 0.3756 - val_loss: 0.3386 - val_mae: 0.3940 - learning_rate: 0.0010
Epoch 20/20
412/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2961 - mae: 0.3768
Epoc

413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2961 - mae: 0.3768 - val_loss: 0.3256 - val_mae: 0.3984 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 20.

CUSTOM TRAINING LOOP
Starting custom training loop...

Epoch 1/10
Loss: 1.0264 - Val Loss: 0.5417

Epoch 2/10
Loss: 0.4304 - Val Loss: 0.4466

Epoch 3/10
Loss: 0.3873 - Val Loss: 0.3963

Epoch 4/10
Loss: 0.3659 - Val Loss: 0.3754

Epoch 5/10
Loss: 0.3544 - Val Loss: 0.3925

Epoch 6/10
Loss: 0.3499 - Val Loss: 0.3850

Epoch 7/10
Loss: 0.3387 - Val Loss: 0.3913

Epoch 8/10
Loss: 0.3407 - Val Loss: 0.4102

Epoch 9/10
Loss: 0.3289 - Val Loss: 0.3496

Epoch 10/10
Loss: 0.3214 - Val Loss: 0.3613

DISTRIBUTED TRAINING SIMULATION
Simulating distributed training...
Number of devices: 1
Distributed model created successfully!

MODEL EVALUATION
Test Loss: 0.3091
Test MAE: 0.3885

SAVING MODEL FOR SERVING


TypeError: this __dict__ descriptor does not support '_DictWrapper' objects